In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [1]:
import os
import numpy as np

from tqdm import tqdm
from pathlib import Path
from datetime import datetime

In [2]:
import torch
import monai
from monai.utils import ensure_tuple_rep

from src.loader import get_dataloader
from src.utils import load_pretrain_model
from src.utils import same_seeds, load_config
from src.SlimUNETR.SlimUNETR import SlimUNETR
from monai.networks.nets import UNet

from main_unlab import calc_metrics_dict
from main_unlab import get_experiment_dir

from accelerate import Accelerator


In [5]:
t1 = torch.zeros((10, 10, 10))
t2 = torch.zeros((10, 10, 10))

In [6]:
torch.stack([t1, t2]).shape

torch.Size([2, 10, 10, 10])

In [7]:
device = torch.device('cuda:1')
torch.cuda.set_device(device)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [8]:
config, data_flag, is_HepaticVessel = load_config()
config.trainer.batch_size = 8
data_flag

'aneurysms'

In [9]:
same_seeds(config.trainer.seed)
image_size = config.trainer.image_size

model = SlimUNETR(**config.slim_unetr)
model.to(device)

train_loader, val_loader, unlab_loader = get_dataloader(config, data_flag)

In [11]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [12]:
from torchinfo import summary

In [19]:
model_u = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(24, 48, 60),
    strides=(2, 1),
    dropout=0.3,
)

In [15]:
X_test = torch.rand((1, 1, 128, 128, 128))

In [22]:
%%timeit
model_u(X_test)

821 ms ± 63.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
summary(model_u, input_data=X_test, col_names=['input_size', 'output_size', 'num_params'])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #
UNet                                                    [1, 1, 128, 128, 128]     [1, 1, 128, 128, 128]     --
├─Sequential: 1-1                                       [1, 1, 128, 128, 128]     [1, 1, 128, 128, 128]     --
│    └─Convolution: 2-1                                 [1, 1, 128, 128, 128]     [1, 24, 64, 64, 64]       --
│    │    └─Conv3d: 3-1                                 [1, 1, 128, 128, 128]     [1, 24, 64, 64, 64]       672
│    │    └─ADN: 3-2                                    [1, 24, 64, 64, 64]       [1, 24, 64, 64, 64]       1
│    └─SkipConnection: 2-2                              [1, 24, 64, 64, 64]       [1, 48, 64, 64, 64]       --
│    │    └─Sequential: 3-3                             [1, 24, 64, 64, 64]       [1, 24, 64, 64, 64]       178,983
│    └─Convolution: 2-3                                 [1, 48, 64, 64, 64]       [1, 1, 128, 128, 128

In [18]:
summary(model, input_data=X_test, col_names=['input_size', 'output_size', 'num_params'])

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
SlimUNETR                                                    [1, 1, 128, 128, 128]     [1, 1, 128, 128, 128]     --
├─Encoder: 1-1                                               [1, 1, 128, 128, 128]     [1, 64, 96]               6,144
│    └─DepthwiseConvLayer: 2-1                               [1, 1, 128, 128, 128]     [1, 24, 32, 32, 32]       --
│    │    └─Conv3d: 3-1                                      [1, 1, 128, 128, 128]     [1, 24, 32, 32, 32]       1,560
│    │    └─GroupNorm: 3-2                                   [1, 24, 32, 32, 32]       [1, 24, 32, 32, 32]       48
│    └─Sequential: 2-2                                       [1, 24, 32, 32, 32]       [1, 24, 32, 32, 32]       --
│    │    └─Block: 3-3                                       [1, 24, 32, 32, 32]       [1, 24, 32, 32, 32]       16,320
│    │    └─Block: 3-4                                   

In [33]:
torch.onnx.export(model, X_test, 'SlimUNETR.onnx', input_names=["features"], output_names=["logits"])

In [13]:
model

UNet(
  (model): Sequential(
    (0): Convolution(
      (conv): Conv3d(1, 24, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
      (adn): ADN(
        (N): InstanceNorm3d(24, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (D): Dropout(p=0.3, inplace=False)
        (A): PReLU(num_parameters=1)
      )
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): Convolution(
          (conv): Conv3d(24, 48, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): InstanceNorm3d(48, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (D): Dropout(p=0.3, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (1): SkipConnection(
          (submodule): Convolution(
            (conv): Conv3d(48, 60, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (adn): ADN(
              (N): InstanceNorm3d(60, eps=1e-05, momentum=

In [7]:
inference = monai.inferers.SlidingWindowInferer(
    roi_size=ensure_tuple_rep(config.trainer.image_size, dim=3),
    overlap=0.5,
    sw_device=device,
    device=device,
)

metrics = {
    "dice_metric": monai.metrics.DiceMetric(
        include_background=True,
        reduction=monai.utils.MetricReduction.MEAN_BATCH,
        get_not_nans=False,
    ),
    # 'hd95_metric': monai.metrics.HausdorffDistanceMetric(percentile=95, include_background=True, reduction=monai.utils.MetricReduction.MEAN_BATCH, get_not_nans=False)
}

post_trans = monai.transforms.Compose(
    [
        monai.transforms.Activations(sigmoid=True),
        monai.transforms.AsDiscrete(threshold=0.5),
    ]
)


In [8]:
logging_dir = Path(os.getcwd()) / "logs" / str(datetime.now()).replace(":", "_")
accelerator = Accelerator(log_with=["tensorboard"], project_dir=logging_dir)
accelerator.init_trackers("seed test")

In [9]:
def evaluate_seed(model, config, data_flag, device):
    
    i = 1
    base_exp_path_save = get_experiment_dir(config, data_flag, root="model_store")
    while config.finetune.checkpoint not in base_exp_path_save.parents[-i].stem: 
        i += 1

    metrics_value = dict()
    seed_list = list(base_exp_path_save.parents[-i].rglob('seed*'))
    for seed in tqdm(seed_list):
        checkpoint = str(list(seed.rglob('best'))[0] / "pytorch_model.bin")
        model = load_pretrain_model(checkpoint, model, verbose=False)
        model.eval()
        
        for i, image_batch in enumerate(val_loader):
            logits = inference(image_batch["image"].to(device), model)
            val_outputs = [post_trans(i) for i in logits]
            for metric_name in metrics:
                metrics[metric_name](y_pred=val_outputs, y=image_batch["label"].to(device))

        _, batch_acc = calc_metrics_dict(
        metrics, accelerator, data_flag, is_train=False
        )

        metrics_value[seed.name] = batch_acc.item()

    return np.mean(list(metrics_value.values()))

In [10]:
evaluate_seed(model, config, data_flag, device)

100%|██████████| 3/3 [00:12<00:00,  4.30s/it]


0.44024479389190674